# Orpheus (600 BCE – 2026 CE) — Go Philology, Concurrent Katabasis & Time-Lapse Lab

Companion executable notebook for the **Orpheus Diachronic Research Archive** (`github.com/randomcase/Orpheus`).
In this Colab notebook, you will:
1. Bootstrap **Go** and load/clone the Orpheus research repository (`go-engine` + `data/`).
2. Run the zero-dependency Go **Multi-System Polytonic Greek & Latin Transliterator** (`ISO 843`, `ALA-LC`, `5th-c. BCE Attic IPA`, `Byzantine Liturgical`, and `Classical vs. Catholic Ecclesiastical Latin`).
3. Execute the Go **Concurrent *Katabasis* Simulation** (`goroutines`, forward-only `chan<-` contracts, and `context.WithCancel` on Motif C331).
4. Plot a **2,600-year Time-Lapse Visualization** (`560 BCE` to `2026 CE`) tracing genre shifts, Catholic/liturgical transformations, and tragic vs. redemptive endings.

In [ ]:
import os, shutil, subprocess, json
from pathlib import Path

REPO_DIR = Path('/content/Orpheus')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/randomcase/Orpheus.git', str(REPO_DIR)], check=False)

if not shutil.which('go'):
    print('Installing Go toolchain for Colab...')
    subprocess.run('curl -fsSL https://go.dev/dl/go1.26.0.linux-amd64.tar.gz -o /tmp/go.tgz && rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz', shell=True, check=True)
    os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']

print(subprocess.check_output(['go', 'version'], text=True).strip())
if (REPO_DIR / 'go-engine').exists():
    res = subprocess.run(['go', 'run', './cmd/orpheus-engine', '--mode=validate', '--data-dir=../data'], cwd=REPO_DIR / 'go-engine', capture_output=True, text=True, check=True)
    print(res.stdout.strip())


In [ ]:
# Multi-system Polytonic Greek & Latin Transliteration powered by Go
import subprocess, json
from pathlib import Path

GO_DIR = Path('/content/Orpheus/go-engine')
SAMPLES = [
    ('polytonic_greek', 'ΟΡΦΑΣ', 'Delphi Sicyonian Metope (c. 565 BCE)'),
    ('polytonic_greek', 'ὀνομάκλυτον Ὀρφήν', 'Ibycus PMG 306 (c. 545 BCE)'),
    ('polytonic_greek', 'Βίος θάνατος βίος · ἀλήθεια · Διόνυσος Ὀρφικοί', 'Olbia Bone Tablet (c. 485 BCE)'),
    ('polytonic_greek', 'Γῆς παῖς εἰμι καὶ Οὐρανοῦ ἀστερόεντος', 'Hipponion Gold Tablet (c. 400 BCE)'),
    ('polytonic_greek', 'ᾆσμα καινόν, ὁ Λόγος ὁ Χριστός', 'Clement Protrepticus 1.3 (c. 195 CE)'),
    ('latin', 'victusque animi respexit', 'Virgil Georgics IV.491 (29 BCE)'),
    ('latin', 'Orpheus Christus est; Eurydice anima humana per fidem reducta.', 'Ovidius Moralizatus (c. 1340 CE)')
]

for script, text, label in SAMPLES:
    out = subprocess.check_output(
        ['go', 'run', './cmd/orpheus-engine', '--mode=transliterate', f'--script={script}', f'--text={text}'],
        cwd=GO_DIR, text=True
    )
    data = json.loads(out)
    print(f'=== {label} ===')
    print('  Original :', data['original_text'])
    if script == 'polytonic_greek':
        print('  ISO 843  :', data.get('iso_843'))
        print('  Attic IPA:', data.get('attic_ipa'))
        print('  Byzantine:', data.get('byzantine_liturgical'))
    else:
        print('  Classical:', data.get('classical_latin'))
        print('  Catholic :', data.get('ecclesiastical_latin'))


In [ ]:
# Concurrent Katabasis Simulation: Goroutines + Forward-Only Channel + Context Cancellation
import subprocess, json
from pathlib import Path

GO_DIR = Path('/content/Orpheus/go-engine')
for look_back in (11, 0):
    out = subprocess.check_output(
        ['go', 'run', './cmd/orpheus-engine', '--mode=katabasis', '--steps=12', f'--look-back-step={look_back}'],
        cwd=GO_DIR, text=True
    )
    sim = json.loads(out)
    print(f"\n--- Simulation (look_back_at_step={look_back}) -> {sim['outcome']} ({sim['duration_micros']} µs) ---")
    if sim.get('hamartia'):
        print('  Hamartia:', sim['hamartia'])
    for ev in sim['events'][-4:]:
        print(f"  [Step {ev['step']}/{ev['total_steps']}] {ev['actor']} ({ev['phase']}): {ev['message']}")


In [ ]:
# 2,600-Year Time-Lapse Visualization (~565 BCE to 2026 CE)
import json
from pathlib import Path
import matplotlib.pyplot as plt

timeline = json.loads(Path('/content/Orpheus/data/01_master_timeline.json').read_text(encoding='utf-8'))
eras = sorted(set(item['era'] for item in timeline), key=lambda e: min(x['year'] for x in timeline if x['era'] == e))
era_y = {era: idx for idx, era in enumerate(eras)}

fig, ax = plt.subplots(figsize=(13, 6), facecolor='#12100e')
ax.set_facecolor('#1b1713')
for item in timeline:
    x = item['year']
    y = era_y[item['era']]
    ax.scatter(x, y, s=110, color='#d6a856', edgecolor='#fff3dc', zorder=3)
    ax.text(x, y + 0.18, f"{item['id']} ({item['display_date']})", color='#eee6d8', fontsize=7.5, ha='center')

ax.set_yticks(range(len(eras)))
ax.set_yticklabels([e.replace('_', ' ') for e in eras], color='#e6c594', fontsize=9)
ax.set_xlabel('Historical Year (Negative = BCE, Positive = CE)', color='#e6c594')
ax.set_title('Orpheus Diachronic Time-Lapse: 565 BCE to 2026 CE', color='#e6c594', fontsize=13, pad=12)
ax.grid(alpha=0.18, color='#c99a49')
ax.tick_params(colors='#b6aa97')
plt.tight_layout()
plt.show()
